# Chapter 1 — Affine Spaces

Companion notebook to Chapter 1 of *Advanced Calculus for Machine Learning*.
It checks the numerical claims of the chapter with PyTorch. Everything runs
on a CPU in a few seconds.

Contents:

1. Moving the origin: points and vectors react differently
2. Affine combinations (Proposition 1.9 and Example 1.10)
3. Affine maps: the matrix stays, the bias moves
4. Points at height one (homogeneous coordinates)
5. Gradient descent on a quadratic is an affine map (Example 1.23)
6. The origin test on optimizers: SGD, momentum, Adam, weight decay
7. Weight decay ends at a barycentre (Example 1.26)
8. Cosine similarity depends on the origin, distance does not
9. Preview of Chapter 2: Adam fails the basis test

In [1]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. Moving the origin

A frame is an origin and a basis. If the origin is moved by a vector $w$
with components $\omega$, the coordinates of a **point** become
$x' = x - \omega$ and the coordinates of a **vector** do not change
(Proposition 1.8).

In [2]:
def point_to_new(x, omega):
    # coordinates of a point after the origin is moved by omega
    return x - omega


def point_to_old(x_new, omega):
    return x_new + omega


omega = torch.tensor([1.0, 1.0])
a, b = torch.tensor([1.0, 2.0]), torch.tensor([3.0, 1.0])
a_new, b_new = point_to_new(a, omega), point_to_new(b, omega)

# the displacement from a to b is the same array in both frames
print("b - a (old frame):", b - a)
print("b - a (new frame):", b_new - a_new)
assert torch.equal(b - a, b_new - a_new)

b - a (old frame): tensor([ 2., -1.])
b - a (new frame): tensor([ 2., -1.])


## 2. Affine combinations

Proposition 1.9: computed on coordinates, $\sum_i\lambda_ix^{(i)}$ is the
coordinate array of a **point** when $\sum_i\lambda_i = 1$, of a **vector**
when $\sum_i\lambda_i = 0$, and of nothing otherwise. The two computations
differ by $(1 - \sum_i\lambda_i)\,w$ when read as points.

In [3]:
def combo(weights, points):
    return sum(l * p for l, p in zip(weights, points))


def check(weights, points, omega):
    # compute in the old frame, and in the new frame
    old = combo(weights, points)
    new = combo(weights, [point_to_new(p, omega) for p in points])
    as_point = torch.allclose(point_to_new(old, omega), new)
    as_vector = torch.allclose(old, new)
    return as_point, as_vector


pts = [torch.randn(3) for _ in range(4)]
om = torch.randn(3)
for name, lam in [("weights sum to 1", [0.5, 0.2, 0.4, -0.1]),
                  ("weights sum to 0", [1.0, -2.0, 1.5, -0.5]),
                  ("weights sum to 2", [1.0, 1.0, 0.5, -0.5])]:
    p, v = check(lam, pts, om)
    print(f"{name}: sum = {sum(lam):+.1f}  point: {p}  vector: {v}")

weights sum to 1: sum = +1.0  point: True  vector: False
weights sum to 0: sum = +0.0  point: False  vector: True
weights sum to 2: sum = +2.0  point: False  vector: False


Example 1.10 with its numbers: $a = (1, 2)$, $b = (3, 1)$, second origin at
$(1, 1)$.

In [4]:
s_from_o = a + b
s_from_o2 = point_to_old(a_new + b_new, omega)
m_from_o = 0.5 * a + 0.5 * b
m_from_o2 = point_to_old(0.5 * a_new + 0.5 * b_new, omega)
print('"a + b" computed from o :', s_from_o.tolist())
print('"a + b" computed from o\':', s_from_o2.tolist())
print("difference              :", (s_from_o2 - s_from_o).tolist(), "= -w")
print("midpoint from o, from o':", m_from_o.tolist(), m_from_o2.tolist())
assert torch.equal(s_from_o, torch.tensor([4.0, 3.0]))
assert torch.equal(s_from_o2, torch.tensor([3.0, 2.0]))
assert torch.equal(m_from_o, m_from_o2)

"a + b" computed from o : [4.0, 3.0]
"a + b" computed from o': [3.0, 2.0]
difference              : [-1.0, -1.0] = -w
midpoint from o, from o': [2.0, 1.5] [2.0, 1.5]


## 3. Affine maps: the matrix stays, the bias moves

A layer `torch.nn.Linear` computes $x\mapsto Wx + b$: an affine map. If the
origin of the input space is moved by $\omega$ (for instance, the inputs are
centred), the same map is written with the same $W$ and the bias
$b + W\omega$.

In [5]:
layer = torch.nn.Linear(4, 3)
W, bias = layer.weight.detach(), layer.bias.detach()
x = torch.randn(5, 4)          # five input points
omega4 = torch.randn(4)        # the origin of the input space moves by omega4

shifted = torch.nn.Linear(4, 3)
with torch.no_grad():
    shifted.weight.copy_(W)                # same matrix
    shifted.bias.copy_(bias + W @ omega4)  # new bias

assert torch.allclose(layer(x), shifted(x - omega4))
print("same outputs with the same W and the bias b + W omega")

# the linear part acts on displacements, and the bias is invisible to them
v = torch.randn(4)
assert torch.allclose(layer(x + v) - layer(x), (W @ v).expand(5, 3))
print("layer(x + v) - layer(x) = W v for every point x")

# composition: linear parts compose, and a stack of affine layers is affine
layer2 = torch.nn.Linear(3, 2)
W2, b2 = layer2.weight.detach(), layer2.bias.detach()
assert torch.allclose(layer2(layer(x)), x @ (W2 @ W).T + (W2 @ bias + b2))
print("two stacked layers = one layer with W2 W1 and W2 b1 + b2")

same outputs with the same W and the bias b + W omega
layer(x + v) - layer(x) = W v for every point x
two stacked layers = one layer with W2 W1 and W2 b1 + b2


A layer **without** bias cannot absorb a change of origin (Exercise 1.6).
We fit $y \approx wx$ by least squares, with the temperature $x$ in degrees
Celsius and then in kelvins, and compare the predictions for the same
physical temperature.

In [6]:
celsius = torch.tensor([[0.0], [10.0], [20.0], [30.0]])
y = torch.tensor([[1.0], [2.0], [2.5], [4.0]])
kelvin = celsius + 273.15


def fit(X, y, with_bias):
    if with_bias:
        X = torch.cat([X, torch.ones(len(X), 1)], dim=1)   # the bias trick
    return torch.linalg.lstsq(X, y).solution


def predict(coef, X, with_bias):
    if with_bias:
        X = torch.cat([X, torch.ones(len(X), 1)], dim=1)
    return X @ coef


t_c = torch.tensor([[25.0]])
for with_bias in (False, True):
    p_c = predict(fit(celsius, y, with_bias), t_c, with_bias)
    p_k = predict(fit(kelvin, y, with_bias), t_c + 273.15, with_bias)
    print(f"bias={with_bias!s:5}  prediction at 25 C: "
          f"fitted in Celsius {p_c.item():.4f}, in kelvins {p_k.item():.4f}")

bias=False  prediction at 25 C: fitted in Celsius 3.3929, in kelvins 2.4963
bias=True   prediction at 25 C: fitted in Celsius 3.3250, in kelvins 3.3250


## 4. Points at height one

Store a point as $(x, 1)$ and a vector as $(v, 0)$. The last coordinate of a
weighted sum is the total weight, and one $(n+1)\times(n+1)$ matrix carries
both the affine map (on points) and its linear part (on vectors).

In [7]:
def as_point(x):
    return torch.cat([x, torch.ones(1)])


def as_vector(v):
    return torch.cat([v, torch.zeros(1)])


def hom(A, beta):
    n_out, n_in = A.shape
    M = torch.zeros(n_out + 1, n_in + 1)
    M[:n_out, :n_in], M[:n_out, -1], M[-1, -1] = A, beta, 1.0
    return M


A1, beta1 = torch.randn(3, 2), torch.randn(3)
A2, beta2 = torch.randn(2, 3), torch.randn(2)
M1, M2 = hom(A1, beta1), hom(A2, beta2)
p, q, u = torch.randn(2), torch.randn(2), torch.randn(2)

print("height of p + u      :", (as_point(p) + as_vector(u))[-1].item())
print("height of q - p      :", (as_point(q) - as_point(p))[-1].item())
print("height of (p + q) / 2:", (0.5 * as_point(p) + 0.5 * as_point(q))[-1].item())
print("height of p + q      :", (as_point(p) + as_point(q))[-1].item())

assert torch.allclose(M1 @ as_point(p), as_point(A1 @ p + beta1))
assert torch.allclose(M1 @ as_vector(u), as_vector(A1 @ u))   # no beta
assert torch.allclose(M2 @ M1, hom(A2 @ A1, A2 @ beta1 + beta2))
print("composition of affine maps = product of the matrices")

height of p + u      : 1.0
height of q - p      : 0.0
height of (p + q) / 2: 1.0
height of p + q      : 2.0
composition of affine maps = product of the matrices


## 5. Gradient descent on a quadratic is an affine map

For $L(x) = \tfrac12x^\top Hx - b^\top x$, one step is
$x\mapsto(I - \eta H)x + \eta b$. Seen from its fixed point $x^*$ (the
solution of $Hx = b$), the map is linear:
$x_k - x^* = (I - \eta H)^k(x_0 - x^*)$ (Proposition 1.22, Example 1.23).

In [8]:
B = torch.randn(3, 3)
H = B @ B.T + torch.eye(3)       # symmetric positive definite
bvec = torch.randn(3)
eta, steps = 0.05, 40

x = torch.randn(3, requires_grad=True)
x0 = x.detach().clone()
opt = torch.optim.SGD([x], lr=eta)
for _ in range(steps):
    opt.zero_grad()
    loss = 0.5 * x @ H @ x - bvec @ x
    loss.backward()
    opt.step()

x_star = torch.linalg.solve(H, bvec)
predicted = torch.linalg.matrix_power(torch.eye(3) - eta * H, steps) @ (x0 - x_star)
print("x_k - x*              :", (x.detach() - x_star).tolist())
print("(I - eta H)^k (x0 - x*):", predicted.tolist())
assert torch.allclose(x.detach() - x_star, predicted)

x_k - x*              : [-0.011562652365000203, -0.005501028105420788, 0.02876392956742721]
(I - eta H)^k (x0 - x*): [-0.011562652365000193, -0.0055010281054207934, 0.028763929567427224]


## 6. The origin test on optimizers

We train the same loss in two frames whose origins differ by $w$: the loss
in the new coordinates is $L'(x') = L(x' + \omega)$, and the starting point
is the same point, $x'_0 = x_0 - \omega$. An update rule passes the origin
test if the two runs visit the same points, that is, if
$x'_k = x_k - \omega$ for all $k$.

In [9]:
def loss_fn(x):
    # any smooth, non-quadratic loss will do
    return torch.log(1 + torch.exp(x[0] - 2 * x[1])) + 0.1 * (x ** 4).sum() \
        + 0.5 * ((x - 1.0) ** 2).sum()


def run(make_opt, x_init, shift, steps=200):
    # run in the frame whose origin is moved by `shift`; return old coordinates
    xp = (x_init - shift).clone().requires_grad_(True)
    opt = make_opt([xp])
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(xp + shift).backward()     # L'(x') = L(x' + omega)
        opt.step()
    return xp.detach() + shift


x_init = torch.tensor([2.0, -1.0, 0.5])
omega3 = torch.tensor([3.0, -2.0, 1.0])
zero = torch.zeros(3)

optimizers = {
    "SGD": lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD + momentum": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam": lambda p: torch.optim.Adam(p, lr=0.05),
    "SGD + weight decay": lambda p: torch.optim.SGD(p, lr=0.05, weight_decay=0.5),
    "AdamW (decoupled decay)": lambda p: torch.optim.AdamW(p, lr=0.05, weight_decay=0.5),
}
for name, make_opt in optimizers.items():
    end_o = run(make_opt, x_init, zero)
    end_o2 = run(make_opt, x_init, omega3)
    gap = (end_o - end_o2).norm().item()
    verdict = "passes" if gap < 1e-8 else "FAILS"
    print(f"{name:24} distance between the two end points: {gap:.2e}  {verdict}")

SGD                      distance between the two end points: 1.57e-16  passes
SGD + momentum           distance between the two end points: 0.00e+00  passes


Adam                     distance between the two end points: 9.49e-16  passes
SGD + weight decay       distance between the two end points: 6.62e-01  FAILS


AdamW (decoupled decay)  distance between the two end points: 1.30e+00  FAILS


After **one** step of SGD with weight decay, the two points differ by exactly
$\eta\lambda w$ (Exercise 1.5).

In [10]:
lr, lam = 0.05, 0.5
make = lambda p: torch.optim.SGD(p, lr=lr, weight_decay=lam)
one_o = run(make, x_init, zero, steps=1)
one_o2 = run(make, x_init, omega3, steps=1)
print("difference after one step:", (one_o2 - one_o).tolist())
print("eta * lambda * omega     :", (lr * lam * omega3).tolist())
assert torch.allclose(one_o2 - one_o, lr * lam * omega3)

difference after one step: [0.07500000000000018, -0.050000000000000155, 0.02499999999999991]
eta * lambda * omega     : [0.07500000000000001, -0.05, 0.025]


## 7. Weight decay ends at a barycentre

Example 1.26: with the field $g(a) = c - a$ (that is, the loss
$\tfrac12\lVert a - c\rVert^2$) and a decay of strength $\lambda$ centred at
a point $o$, training converges to
$a^* = \tfrac1{1+\lambda}c + \tfrac\lambda{1+\lambda}o$. With $\lambda = 1$
and $c = (3, 1)$ the end point is the midpoint of $c$ and the centre of
decay. `weight_decay` in PyTorch always decays towards the zero array, so a
decay centred elsewhere is obtained by running in the frame whose origin is
that centre.

In [11]:
c = torch.tensor([3.0, 1.0])
a0 = torch.tensor([0.5, 2.5])
lam, eta = 1.0, 0.25


def train_with_decay_centred_at(centre, steps=60):
    xp = (a0 - centre).clone().requires_grad_(True)   # frame with origin `centre`
    opt = torch.optim.SGD([xp], lr=eta, weight_decay=lam)
    path = [a0.clone()]
    for _ in range(steps):
        opt.zero_grad()
        (0.5 * ((xp + centre - c) ** 2).sum()).backward()
        opt.step()
        path.append(xp.detach() + centre)
    return torch.stack(path)


for centre in (torch.tensor([0.0, 0.0]), torch.tensor([2.0, 2.0])):
    path = train_with_decay_centred_at(centre)
    expected = c / (1 + lam) + lam / (1 + lam) * centre
    print(f"decay centred at {centre.tolist()}: ends at "
          f"{[round(v, 6) for v in path[-1].tolist()]}, "
          f"barycentre {expected.tolist()}")
    print("   first iterates:", [[round(v, 4) for v in p] for p in path[:4].tolist()])
    assert torch.allclose(path[-1], expected)

decay centred at [0.0, 0.0]: ends at [1.5, 0.5], barycentre [1.5, 0.5]
   first iterates: [[0.5, 2.5], [1.0, 1.5], [1.25, 1.0], [1.375, 0.75]]
decay centred at [2.0, 2.0]: ends at [2.5, 1.5], barycentre [2.5, 1.5]
   first iterates: [[0.5, 2.5], [1.5, 2.0], [2.0, 1.75], [2.25, 1.625]]


The first iterates are the ones plotted in Figure 1.5: the linear part of
the map is $(1 - \eta(1 + \lambda))\,\mathrm{id} = \tfrac12\,\mathrm{id}$,
so the distance to the fixed point is halved at each step.

## 8. Cosine similarity depends on the origin, distance does not

The cosine similarity of two **points** is computed on the vectors that
join the origin to them. Move the origin and it changes; the distance
between the two points does not.

In [12]:
p1, p2 = torch.tensor([1.0, 0.0]), torch.tensor([0.0, 1.0])
cos = torch.nn.functional.cosine_similarity
for origin in (torch.tensor([0.0, 0.0]), torch.tensor([-1.0, -1.0])):
    q1, q2 = p1 - origin, p2 - origin
    print(f"origin at {origin.tolist()}: cosine = {cos(q1, q2, dim=0).item():.4f}, "
          f"distance = {(q1 - q2).norm().item():.4f}")

origin at [0.0, 0.0]: cosine = 0.0000, distance = 1.4142
origin at [-1.0, -1.0]: cosine = 0.8000, distance = 1.4142


## 9. Preview of Chapter 2: Adam fails the basis test

Adam passed the origin test above. It does not pass the *basis* test, even
for a rotation. On the linear loss $L(x) = g\cdot x$ with $g = (2, 1)$, its
first step is $-\eta\,\mathrm{sign}(g) = -\eta(1, 1)$. In the orthonormal
basis rotated by $45^\circ$, the same loss gives the step
$-\eta(\sqrt2, 0)$ once expressed in the original basis: a different
length and a different direction. Plain gradient descent gives the same
step in both bases.

In [13]:
g = torch.tensor([2.0, 1.0])
r = 1 / math.sqrt(2)
Q = torch.tensor([[r, -r], [r, r]])     # columns: the rotated orthonormal basis


def first_step(make_opt, basis):
    # coordinates z in `basis`: x = basis @ z, and L(x) = g . x
    z = torch.zeros(2, requires_grad=True)
    opt = make_opt([z])
    (g @ (basis @ z)).backward()
    opt.step()
    return basis @ z.detach()           # the step, in the standard basis


eta = 0.1
for name, make_opt in [("SGD ", lambda p: torch.optim.SGD(p, lr=eta)),
                       ("Adam", lambda p: torch.optim.Adam(p, lr=eta, eps=0.0))]:
    s_std = first_step(make_opt, torch.eye(2))
    s_rot = first_step(make_opt, Q)
    fmt = lambda t: [round(v, 4) + 0.0 for v in t.tolist()]   # no "-0.0"
    print(f"{name}: step in the standard basis {fmt(s_std)}, "
          f"in the rotated basis {fmt(s_rot)}")

SGD : step in the standard basis [-0.2, -0.1], in the rotated basis [-0.2, -0.1]
Adam: step in the standard basis [-0.1, -0.1], in the rotated basis [-0.1414, 0.0]
